# 06_sensitivity_dm
Sensitivity analyses and a parallel diabetes (DM) target.
Three sensitivity questions are addressed alongside the primary HTN results:
(S1) Are the axis-1 realisability gaps an artefact of DiCE's default search? We
regenerate recourse with a strong proximity/sparsity preference and compare the
prescribed-change distribution against the baseline recourse.
(S2) Is the axis-3 utilisation contrast driven by a healthy-adherer effect? We
re-estimate axis-2 with baseline utilisation added to the propensity model and
report how the treatment effect moves.
(S3) Do the findings replicate for a second target? We rerun the target-trial
(axis 2), the realisability gap (axis 1) and the equity gradient (axis 5) for DM.
All figures greyscale, dpi 600, png+pdf; tables to results/tables.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import joblib, statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import dice_ml
from scipy.stats import mannwhitneyu
rng=np.random.default_rng(42); RNG=42

panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
tr =pd.read_parquet(os.path.join(DATA_DIR,"transitions_1y.parquet"))
tr2=pd.read_parquet(os.path.join(DATA_DIR,"transitions_2y.parquet"))
for df in (tr,tr2): df["d_BMI"]=df["BMI_t1"]-df["BMI_t0"]
print("loaded panel/transitions")

loaded panel/transitions


In [3]:
# ============================================================
# S1: recourse under a STRONG proximity/sparsity preference
# ============================================================
ACT=["BMI","PA_REG","PA_WALK","ALC_FREQ"]; COV=["age","SEX","EDU","H_INC_TOT"]; feat=ACT+COV
def prep_frame(target_dx):
    d=panel.dropna(subset=feat+[target_dx]).copy()
    d["SEX"]=(d["SEX"]=="M").astype(int)
    for c in ["PA_REG","EDU"]: d[c]=pd.to_numeric(d[c],errors="coerce")
    d=d.dropna(subset=feat+[target_dx])
    d=d.sort_values("year").groupby(KEY,as_index=False).first()
    return d
d=prep_frame("HTN_dx")
X=d[feat].astype(float); y=d["HTN_dx"].astype(int)
mA=GradientBoostingClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
                              subsample=0.9,random_state=RNG).fit(X,y)
dd=pd.concat([X,y.rename("HTN")],axis=1)
data_dice=dice_ml.Data(dataframe=dd,continuous_features=[c for c in feat if c!="SEX"],outcome_name="HTN")
exp=dice_ml.Dice(data_dice,dice_ml.Model(model=mA,backend="sklearn",model_type="classifier"),method="genetic")
p=mA.predict_proba(X)[:,1]; thr=np.quantile(p,0.70)
q=X[p>=thr].sample(min(150,int((p>=thr).sum())),random_state=RNG)[feat].astype(float).reset_index(drop=True)

def gen(row_df, prox, spars):
    r=row_df.iloc[0]
    pr={"BMI":[16.0,float(r["BMI"])],"PA_WALK":[float(r["PA_WALK"]),7.0],
        "ALC_FREQ":[0.0,float(r["ALC_FREQ"])],"PA_REG":[float(r["PA_REG"]),1.0]}
    for k,(lo,hi) in list(pr.items()):
        if hi<=lo: pr[k]=[lo,hi+1e-6] if k in ("PA_WALK","PA_REG") else [max(lo-1e-6,0),hi]
    return exp.generate_counterfactuals(row_df,total_CFs=1,desired_class=0,features_to_vary=ACT,
            permitted_range=pr,proximity_weight=prox,sparsity_weight=spars)

def collect(prox,spars,label):
    out=[]
    for i in range(len(q)):
        try:
            cfdf=gen(q.iloc[[i]],prox,spars).cf_examples_list[0].final_cfs_df
            if cfdf is not None and len(cfdf):
                out.append(float(cfdf.iloc[0]["BMI"])-float(q.iloc[i]["BMI"]))
        except Exception: pass
    s=pd.Series(out,name=label); return s

base=collect(1.5,1.0,"baseline")     # same as notebook 02
strong=collect(4.0,2.0,"strong_prox")
print("valid CFs  baseline:",len(base)," strong:",len(strong))
s1=pd.DataFrame({"setting":["baseline","strong_prox"],
    "n":[len(base),len(strong)],
    "median_prescribed_BMI_reduction":[round(-base.median(),2),round(-strong.median(),2)],
    "q90_prescribed_reduction":[round(-base.quantile(0.10),2),round(-strong.quantile(0.10),2)]})
savetable(s1,"t06_s1_recourse_magnitude", index=False)
print(s1.to_string(index=False))

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.70it/s]

100%|██████████| 1/1 [00:00<00:00,  6.63it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  7.11it/s]

...

100%|██████████| 1/1 [00:00<00:00,  7.03it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  7.20it/s]

100%|██████████| 1/1 [00:00<00:00,  7.13it/s]

valid CFs  baseline: 132  strong: 126
saved: t06_s1_recourse_magnitude.csv
    setting   n  median_prescribed_BMI_reduction  q90_prescribed_reduction
   baseline 132                             3.59                      9.03
strong_prox 126                             3.72                      9.32


In [4]:
# S1 figure: prescribed BMI-reduction distribution, baseline vs strong proximity
fig,ax=plt.subplots(figsize=(6.0,4.0))
for s,gc,lab in [(-base,"#999999","baseline"),(-strong,"#333333","strong proximity")]:
    sns.kdeplot(s.clip(0,20),ax=ax,color=gc,lw=1.6,label=lab,fill=False)
# empirical population 90th-pct realised reduction for reference
atr=tr[tr["HTN_atrisk"]==1]; popb=float((-atr["d_BMI"]).quantile(0.90))
ax.axvline(popb,color="#000000",lw=1.0,ls="--",label=f"pop q90 realised = {popb:.1f}")
ax.set_xlabel("Prescribed BMI reduction"); ax.set_ylabel("Density")
ax.legend(frameon=False,fontsize=8)
savefig(fig,"f06_s1_recourse_magnitude"); plt.close(fig)
print("S1 figure saved; population q90 realised reduction =",round(popb,2))

saved: f06_s1_recourse_magnitude.png / f06_s1_recourse_magnitude.pdf
S1 figure saved; population q90 realised reduction = 1.06


In [5]:
# ============================================================
# Shared target-trial estimator (used by S2 and the DM target)
# ============================================================
def build_trial(df, target, tau=None):
    d=df[df[f"{target}_atrisk"]==1].copy()
    need=["BMI_t0","BMI_t1",f"{target}_onset","age_t0","SEX_t0","H_INC_TOT_t0"]
    d=d.dropna(subset=need)
    red=-(d["BMI_t1"]-d["BMI_t0"])
    if tau is None: tau=float(red[red>0].quantile(0.75))
    d["TREAT"]=(red>=tau).astype(int)
    d["male"]=(d["SEX_t0"]=="M").astype(int)
    d["onset"]=d[f"{target}_onset"].astype(int)
    d["bmi0"]=d["BMI_t0"]; d["age0"]=d["age_t0"]
    d["inc0"]=d["H_INC_TOT_t0"].fillna(d["H_INC_TOT_t0"].median())
    d["year0"]=d["year0"].astype("category")
    return d, tau

def ipw_weights(d, ps_cols):
    Xps=d[ps_cols].copy()
    for c in ps_cols: Xps[c]=Xps[c].fillna(Xps[c].median())
    e=Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=1000))]).fit(Xps,d["TREAT"]).predict_proba(Xps)[:,1]
    e=np.clip(e,0.02,0.98); pt=d["TREAT"].mean()
    return np.where(d["TREAT"]==1, pt/e,(1-pt)/(1-e))

def fit_axis2(d, sw):
    m=smf.glm("onset ~ TREAT + bmi0 + age0 + male + C(year0)",data=d,
              family=sm.families.Binomial(),freq_weights=sw).fit(cov_type="HC1")
    orT=np.exp(m.params["TREAT"]); ci=np.exp(m.conf_int().loc["TREAT"])
    return m, orT, ci

def ipw_rd(d,sw):
    t=d["TREAT"].values; yv=d["onset"].values
    r1=np.sum(sw*t*yv)/np.sum(sw*t); r0=np.sum(sw*(1-t)*yv)/np.sum(sw*(1-t))
    return r1-r0
print("estimator helpers ready")

estimator helpers ready


In [6]:
# ============================================================
# S2: healthy-adherer control -- add baseline utilisation to the PS model
# ============================================================
# Baseline (t0) utilisation must be attached; transitions carry only t1 utilisation,
# so we merge t0 utilisation from the long panel by person + baseline year.
util0=panel[[KEY,"year","OUGUN","INGUN","OUOOP_1"]].rename(
        columns={"year":"year0","OUGUN":"OUGUN_t0","INGUN":"INGUN_t0","OUOOP_1":"OUOOP_1_t0"})
trU=tr.merge(util0,on=[KEY,"year0"],how="left")

dH,tauH=build_trial(trU,"HTN")
base_ps=["bmi0","age0","male","inc0"]
adj_ps =base_ps+["OUGUN_t0","INGUN_t0","OUOOP_1_t0"]
for c in ["OUGUN_t0","INGUN_t0","OUOOP_1_t0"]:
    dH[c]=pd.to_numeric(dH[c],errors="coerce")

sw_base=ipw_weights(dH,base_ps)
sw_adj =ipw_weights(dH,adj_ps)
_,or_b,ci_b=fit_axis2(dH,sw_base)
_,or_a,ci_a=fit_axis2(dH,sw_adj)
rd_b=ipw_rd(dH,sw_base); rd_a=ipw_rd(dH,sw_adj)
s2=pd.DataFrame([
 {"model":"baseline PS","OR":round(or_b,3),"OR_lo":round(ci_b[0],3),"OR_hi":round(ci_b[1],3),"risk_diff_pp":round(rd_b*100,3)},
 {"model":"PS + baseline utilisation","OR":round(or_a,3),"OR_lo":round(ci_a[0],3),"OR_hi":round(ci_a[1],3),"risk_diff_pp":round(rd_a*100,3)},
])
savetable(s2,"t06_s2_healthy_adherer", index=False)
print(s2.to_string(index=False))

saved: t06_s2_healthy_adherer.csv
                    model    OR  OR_lo  OR_hi  risk_diff_pp
              baseline PS 0.608  0.450  0.822        -1.009
PS + baseline utilisation 0.599  0.442  0.810        -1.042


In [7]:
# ============================================================
# DM target: axis 2 (1y + 2y), axis 1 gap, axis 5 equity
# ============================================================
# axis 2 (1-year)
dDM,tauDM=build_trial(tr,"DM")
swDM=ipw_weights(dDM,["bmi0","age0","male","inc0"])
mDM,orDM,ciDM=fit_axis2(dDM,swDM); rdDM=ipw_rd(dDM,swDM)
# axis 2 (2-year, cloglog discrete hazard)
dDM2,_=build_trial(tr2,"DM",tau=tauDM)
mDM2=smf.glm("onset ~ TREAT + bmi0 + age0 + male + C(year0)",data=dDM2,
             family=sm.families.Binomial(sm.families.links.CLogLog())).fit(cov_type="HC1")
hrDM=np.exp(mDM2.params["TREAT"]); ciDM2=np.exp(mDM2.conf_int().loc["TREAT"])
dm2=pd.DataFrame([
 {"horizon":"1y","estimator":"IPW logistic","effect":"OR","point":round(orDM,3),"lo":round(ciDM[0],3),"hi":round(ciDM[1],3)},
 {"horizon":"1y","estimator":"IPW risk diff","effect":"pp","point":round(rdDM*100,3),"lo":np.nan,"hi":np.nan},
 {"horizon":"2y","estimator":"cloglog","effect":"HR-like","point":round(hrDM,3),"lo":round(ciDM2[0],3),"hi":round(ciDM2[1],3)},
])
savetable(dm2,"t06_dm_axis2", index=False)
print("DM axis2:\n",dm2.to_string(index=False))

saved: t06_dm_axis2.csv
DM axis2:
 horizon     estimator  effect  point    lo    hi
     1y  IPW logistic      OR  1.368 1.006 1.859
     1y IPW risk diff      pp  0.438   NaN   NaN
     2y       cloglog HR-like  1.454 1.188 1.781


In [8]:
# DM axis 1: prescribed vs realised (reuse HTN recourse magnitude as prescription proxy,
# since recourse is model-driven; realised change distribution is target-agnostic but
# evaluated on the DM at-risk population)
atrDM=tr[tr["DM_atrisk"]==1].copy(); atrDM["red_BMI"]=-(atrDM["BMI_t1"]-atrDM["BMI_t0"])
med_presc=float((-base).median())   # baseline recourse median prescribed reduction
ach=(atrDM["red_BMI"]>=med_presc).mean()
popb_dm=float(atrDM["red_BMI"].quantile(0.90))
dm1=pd.DataFrame([{"target":"DM","median_prescribed_reduction":round(med_presc,2),
                   "pop_achievement_at_median":round(float(ach),4),
                   "pop_q90_realised_reduction":round(popb_dm,2)}])
savetable(dm1,"t06_dm_axis1", index=False)
print(dm1.to_string(index=False))

saved: t06_dm_axis1.csv
target  median_prescribed_reduction  pop_achievement_at_median  pop_q90_realised_reduction
    DM                         3.59                     0.0092                        1.07


In [9]:
# DM axis 5: recourse-consistent attainment by age band + income quintile
tauA=float(atrDM["red_BMI"][atrDM["red_BMI"]>0].quantile(0.75))
atrDM["attain"]=(atrDM["red_BMI"]>=tauA).astype(int)
def rate_by(df,col):
    g=df.dropna(subset=[col]).groupby(col,observed=True)["attain"].agg(["mean","size"]).reset_index()
    g.columns=[ "level","attain_rate","n"]; g.insert(0,"stratum",col)
    g["attain_rate"]=g["attain_rate"].round(3); return g
dm5=pd.concat([rate_by(atrDM,c) for c in ["AGEG_t0","INCQ_t0"] if c in atrDM.columns],ignore_index=True)
savetable(dm5,"t06_dm_axis5_attainment", index=False)
print(dm5.to_string(index=False))

saved: t06_dm_axis5_attainment.csv
stratum level  attain_rate     n
AGEG_t0 19-29        0.089  3066
AGEG_t0 30-39        0.099  4143
AGEG_t0 40-49        0.072  6178
AGEG_t0 50-59        0.068  6825
AGEG_t0 60-69        0.066  9083
AGEG_t0   70+        0.080 10505
INCQ_t0    Q1        0.086  8513
INCQ_t0    Q2        0.073  8551
INCQ_t0    Q3        0.070  8095
INCQ_t0    Q4        0.078  7470
INCQ_t0    Q5        0.071  7003


In [10]:
# Combined figure: adjusted onset risk (treated vs control) for HTN and DM, 1-year
def adj_rates(model,d):
    d1=d.copy(); d1["TREAT"]=1; d0=d.copy(); d0["TREAT"]=0
    return float(model.predict(d1).mean()), float(model.predict(d0).mean())
mH,_,_=fit_axis2(dH,sw_base); tH,cH=adj_rates(mH,dH); tD,cD=adj_rates(mDM,dDM)
fig,ax=plt.subplots(figsize=(5.4,4.0))
xs=np.arange(2); w=0.35
ax.bar(xs-w/2,[cH*100,cD*100],width=w,label="control",color="#bbbbbb",edgecolor="black",linewidth=0.5)
ax.bar(xs+w/2,[tH*100,tD*100],width=w,label="treated",color="#555555",edgecolor="black",linewidth=0.5)
ax.set_xticks(xs); ax.set_xticklabels(["HTN","DM"])
ax.set_ylabel("Adjusted 1-year onset risk (%)"); ax.legend(frameon=False,fontsize=8)
savefig(fig,"f06_dm_htn_adjusted_onset"); plt.close(fig)
print(f"adjusted onset HTN: treated {tH*100:.2f}% vs control {cH*100:.2f}% | DM: treated {tD*100:.2f}% vs control {cD*100:.2f}%")

saved: f06_dm_htn_adjusted_onset.png / f06_dm_htn_adjusted_onset.pdf
adjusted onset HTN: treated 1.90% vs control 3.06% | DM: treated 1.61% vs control 1.19%


In [11]:
# Consolidated sensitivity/DM headline
head=pd.DataFrame([
 {"item":"S1 baseline median prescribed BMI reduction","value":round(-base.median(),2)},
 {"item":"S1 strong-prox median prescribed BMI reduction","value":round(-strong.median(),2)},
 {"item":"S1 population q90 realised reduction","value":round(popb,2)},
 {"item":"S2 HTN OR baseline PS","value":round(or_b,3)},
 {"item":"S2 HTN OR + baseline utilisation","value":round(or_a,3)},
 {"item":"DM 1y treatment OR","value":round(orDM,3)},
 {"item":"DM 1y IPW risk diff (pp)","value":round(rdDM*100,3)},
 {"item":"DM pop achievement at median prescription","value":round(float(ach),4)},
])
savetable(head,"t06_headline_sensitivity_dm", index=False)
print(head.to_string(index=False))

saved: t06_headline_sensitivity_dm.csv
                                          item  value
   S1 baseline median prescribed BMI reduction 3.5900
S1 strong-prox median prescribed BMI reduction 3.7200
          S1 population q90 realised reduction 1.0600
                         S2 HTN OR baseline PS 0.6080
              S2 HTN OR + baseline utilisation 0.5990
                            DM 1y treatment OR 1.3680
                      DM 1y IPW risk diff (pp) 0.4380
     DM pop achievement at median prescription 0.0092
